# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_dwh;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;')

## Đọc data từ SQL Server

In [4]:
query_phieumuon = """SELECT TOP (100) TL.ID_tai_lieu, XG.ID_xep_gia, Ma_tai_lieu, Ngay_giao_dich 
                        FROM olap.DIM_Tai_lieu TL
                        JOIN olap.DIM_Xep_gia XG ON TL.ID_tai_lieu =  XG.ID_tai_lieu"""
df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)
print(df_phieumuon)

    ID_tai_lieu  ID_xep_gia  Ma_tai_lieu  Ngay_giao_dich
0             0           0            0               0
1            31      182098  SK020000032        20060427
2            31      182099  SK020000032        20060427
3            31      182100  SK020000032        20060427
4            31      182101  SK020000032        20060427
..          ...         ...          ...             ...
95          191      182426  SK020000195        20070417
96          191      182427  SK020000195        20070417
97          191      182428  SK020000195        20070417
98          191      182429  SK020000195        20070417
99          191      182430  SK020000195        20070417

[100 rows x 4 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_8492\1167354447.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)


# Xử lý code

In [5]:
so_ban_sach = df_phieumuon.groupby(['ID_tai_lieu', 'Ma_tai_lieu', 'Ngay_giao_dich'])['ID_xep_gia'].count().reset_index()
so_ban_sach = so_ban_sach.rename(columns={'ID_xep_gia': 'So_ban_sach'})
print(so_ban_sach)

    ID_tai_lieu  Ma_tai_lieu  Ngay_giao_dich  So_ban_sach
0             0            0               0            1
1            26    SKV000004        20070417            1
2            31  SK020000032        20060427            5
3            38  SK020000040        20060427            5
4            41  SK020000044        20070417            5
..          ...          ...             ...          ...
58          185  SK020000188        20231129            1
59          186  SK020000189        20231129            1
60          188  SK020000192        20070417            3
61          189  SK020000193        20130801            2
62          191  SK020000195        20070417            5

[63 rows x 4 columns]


## Load data

### [Nếu cần] Clear bảng

In [3]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.FACT_Tai_lieu"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [8]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.FACT_Tai_lieu (ID_tai_lieu,
                                                Ma_xep_gia, 
                                                ID_date, 
                                                So_ban_sach)
                VALUES (?, ?, ?, ?)"""
for index, row in so_ban_sach.iterrows():
   # Trích xuất giá trị từ các cột
    values = (row['ID_tai_lieu'],
              row['Ma_tai_lieu'],
              row['Ngay_giao_dich'],
              row['So_ban_sach'])  # Nếu cột này có tên đúng
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()